# Calibration du scaling — moteur custom vs ×N naïf

**Objectif.** Visualiser où le moteur de scaling déterministe (`app/scaling/`)
**diverge** d'une multiplication naïve `×N` (comportement Cooklang binaire, D6)
et où il l'**égale**. C'est l'outil de la boucle de calibration **F2** et la
réponse au risque **R3** (« coefficients faux »).

- **×N naïf** = `qty × k` pour *toutes* les quantités, indistinctement.
- **moteur** = sortie de `scale(...)` / `scale_eggs(...)` / `scale_time(...)`
  (sous-linéaire pour le sel/les épices, discret pour les œufs, géométrique pour
  le temps, figé pour les températures).
- **écart** = `moteur − ×N naïf` (un écart non nul signale là où la cuisine réelle
  s'écarte de la règle de trois).

> ⚠️ **Disclaimer (cf. `table-scaling-sale.json` → `meta.disclaimer`)** : les
> coefficients sont des valeurs **de DÉPART** issues de la recherche + physique,
> **à CALIBRER** par des tests réels (×2 / ÷2). Ne pas les présenter comme exacts.

> 🧪 **Amorce (sprint 1).** Ce notebook tourne ici sur une **petite liste codée
> en dur** d'ingrédients représentatifs : il ne dépend PAS du corpus S0.6. La
> version *complète* (boucle sur les 10-15 recettes `recipes/` issues de S0.6,
> comparaison ×2 / ÷2 avec de la cuisine réelle) sera finalisée **après S0.6**
> (sprint 2) — voir la cellule de clôture.


In [ ]:
# Le moteur est pur/déterministe : aucun I/O réseau, aucun appel LLM ici.
# On importe depuis la racine du repo (lancer Jupyter depuis la racine, ou
# ajuster sys.path comme ci-dessous pour une exécution « Run All » robuste).
import sys
from pathlib import Path

# Rend le notebook ré-exécutable quel que soit le cwd : on remonte jusqu'à la
# racine du repo (dossier contenant `app/`).
_here = Path.cwd()
for _root in (_here, *_here.parents):
    if (_root / "app" / "scaling" / "engine.py").exists():
        if str(_root) not in sys.path:
            sys.path.insert(0, str(_root))
        break

from app.scaling.engine import scale, scale_eggs, scale_time  # noqa: E402
from app.scaling.table import classify, load_table  # noqa: E402

_meta = load_table().get("meta", {})
print("Table :", _meta.get("name"), "v" + str(_meta.get("version")))
print("Disclaimer :", _meta.get("disclaimer"))


In [ ]:
# Panel d'ingrédients représentatifs (un par type de scaling), codé en dur pour
# l'amorce. (name, qty, unit) — la qté est arbitraire mais réaliste.
PANEL = [
    ("sel", 200, "g"),      # sublinear (coeff 0.75) -> réserve 10 %
    ("tomate", 200, "g"),   # linear (défaut)
    ("ail", 30, "g"),       # sublinear (aromate puissant)
    ("piment", 5, "g"),     # sublinear + flag non-linéarité
    ("oeuf", 3, "u"),       # discrete (œufs liants, RZ1)
    ("temps", 20, "min"),   # geometric (k^0.6667)
]


def moteur_value(name: str, qty: float, unit: str, k: float) -> float:
    """Valeur « moteur » pour un ingrédient du panel, en aiguillant vers la
    bonne fonction selon le type de scaling lu dans la table."""
    rule = classify(name)
    if rule.type == "discrete":          # œufs : on compare le NOMBRE d'œufs
        return float(scale_eggs(int(qty), k).whole_eggs)
    if rule.type == "geometric":         # temps : minutes scalées
        return scale_time(qty, k).minutes
    return scale(name, qty, unit, k).value  # linear / sublinear / fixed


def comparer(k: float) -> list[dict]:
    """Construit la table comparative qty_base | ×N_naïf | moteur | écart
    pour un facteur k donné, sans dépendance lourde (liste de dict)."""
    rows = []
    for name, qty, unit in PANEL:
        naif = qty * k
        moteur = moteur_value(name, qty, unit, k)
        rows.append(
            {
                "ingrédient": name,
                "type": classify(name).type,
                "qty_base": qty,
                "×N_naïf": round(naif, 3),
                "moteur": round(moteur, 3),
                "écart": round(moteur - naif, 3),
            }
        )
    return rows


def afficher(rows: list[dict]) -> None:
    """Rend la table en texte aligné (pas de pandas requis)."""
    cols = ["ingrédient", "type", "qty_base", "×N_naïf", "moteur", "écart"]
    widths = {c: max(len(c), *(len(str(r[c])) for r in rows)) for c in cols}
    header = " | ".join(c.ljust(widths[c]) for c in cols)
    print(header)
    print("-" * len(header))
    for r in rows:
        print(" | ".join(str(r[c]).ljust(widths[c]) for c in cols))


In [ ]:
# ×2 : doublement. C'est le cas canonique. On attend un écart ~nul pour le
# linéaire (tomate) et NÉGATIF pour le sous-linéaire (sel, ail, piment) et le
# temps (le moteur scale MOINS que la règle de trois).
print("=== Facteur k = 2.0 (×2) ===")
afficher(comparer(2.0))


In [ ]:
# ÷2 et ×4 : on vérifie que la divergence est cohérente dans les deux sens.
for k in (0.5, 4.0):
    label = "÷2" if k == 0.5 else "×4"
    print(f"\n=== Facteur k = {k} ({label}) ===")
    afficher(comparer(k))


In [ ]:
# Lecture rapide (amorce) : l'écart est ~0 pour la tomate (linéaire == naïf),
# et s'éloigne de 0 pour sel/ail/piment/temps. À ×2, le sel passe de 400 (naïf)
# à ~336 (moteur) : c'est exactement la divergence que la calibration F2 doit
# valider sur de la cuisine réelle.
rows = comparer(2.0)
ecart_max = max(rows, key=lambda r: abs(r["écart"]))
print("Plus gros écart à ×2 :", ecart_max["ingrédient"], "->", ecart_max["écart"])
assert abs(next(r for r in rows if r["ingrédient"] == "tomate")["écart"]) < 1e-9, \
    "le linéaire doit égaler le ×N naïf"
print("OK — le linéaire égale le ×N naïf, le sous-linéaire/géométrique diverge.")


## Suite : version complète après S0.6

Cette amorce compare le moteur au `×N` naïf sur une **liste figée**
d'ingrédients. La version **complète** branchera le **corpus réel** :

- **S0.6** fournit 10-15 recettes italiennes (`recipes/`, format `cook.md`).
- On bouclera sur ces recettes pour comparer `moteur` vs `×N naïf` **par
  ingrédient réel**, aux facteurs **×2 / ÷2** (cuisine testée en vrai).
- Les écarts alimenteront la **boucle de calibration F2** (ajustement des
  coefficients de `docs/scaling/table-scaling-sale.json`) et le **jeu de
  validation E3**.

📎 Références : `docs/recherche-ouverte.md` §C (calibration) ;
`docs/architecture.md` §11 (ligne « Calibration scaling (moteur vs ×N naïf) »),
§13 R3 ; `table-scaling-sale.json` → `meta.disclaimer`.


## Version complète (E3) — sur le corpus réel S0.6

Cette section **prend le relais** de l'amorce ci-dessus (qui tournait sur une
liste codée en dur). Elle branche le **corpus réel** : les 13 recettes italiennes
`recipes/*.cook` (S0.6). Pour chaque ingrédient réel, on compare le **moteur**
(`scale`/`scale_eggs`/`scale_time`) au **×N naïf**, aux facteurs **×2 / ÷2 / ×4**.

- **Pur / offline** : le moteur est déterministe ; le corpus est lu localement.
  Aucun réseau, aucun LLM, aucune dépendance Epicure/Generator/Knowledge.
- **Parsing `.cook`** : on **recopie** les regex éprouvées de
  `recipes/_check_cook.py` / `app/generator/cooklang.py` (`recipes/` n'est pas un
  package importable).

> Ce jeu **alimente** la calibration **F2 / RZ2** (il chiffre l'écart sur de
> vraies recettes) mais ne la **résout pas** : les coefficients de
> `table-scaling-sale.json` restent « de départ, à calibrer F2 » (RZ2 `open`).

In [ ]:
# Parsing du corpus recipes/*.cook — regex RECOPIÉES de recipes/_check_cook.py
# (recipes/ n'est pas un package importable). Stdlib seule.
import re  # noqa: E402
from pathlib import Path  # noqa: E402

RECIPES_DIR = next(
    (p / "recipes") for p in (_here, *_here.parents) if (p / "recipes").is_dir()
)

# Décompose @nom{[=]qty%unité} ; le verrou `=` marque une quantité figée (fixed).
RE_INGREDIENT_PARTS = re.compile(r"@([^@#~\n]+?)\{(=?)([^}]*?)%([^}]*?)\}")
RE_NUMBER = re.compile(r"^\s*([0-9]+(?:[.,][0-9]+)?|[0-9]+/[0-9]+)\s*$")


def _to_float(raw):
    """Quantité textuelle -> float (robuste : « 1,5 », « 1/2 ») ou None (« q.s. »)."""
    m = RE_NUMBER.match(raw)
    if not m:
        return None
    tok = m.group(1)
    if "/" in tok:
        num, _, den = tok.partition("/")
        return float(num) / float(den)
    return float(tok.replace(",", "."))


def parse_corpus():
    """{nom_fichier: [{name, qty, unit, locked}]} — déterministe, lecture seule."""
    corpus = {}
    for path in sorted(RECIPES_DIR.glob("*.cook")):
        text = path.read_text(encoding="utf-8")
        rows = []
        for mt in RE_INGREDIENT_PARTS.finditer(text):
            qty = _to_float(mt.group(3))
            if qty is None:
                continue
            rows.append({
                "name": mt.group(1).strip(),
                "qty": qty,
                "unit": mt.group(4).strip(),
                "locked": mt.group(2) == "=",
            })
        corpus[path.name] = rows
    return corpus


CORPUS = parse_corpus()
ALL = [dict(r, recipe=name) for name, rows in CORPUS.items() for r in rows]
print(f"Corpus : {len(CORPUS)} recettes, {len(ALL)} ingrédients numériques extraits.")

from collections import Counter  # noqa: E402
par_type = Counter(classify(r["name"]).type for r in ALL)
print("Répartition par type de scaling :", dict(par_type))

In [ ]:
# Comparaison moteur vs ×N naïf, PAR INGRÉDIENT du corpus, pour un facteur k.
# On aiguille selon le type lu dans la table (linéaire/sous-linéaire/œufs/temps).
def moteur_corpus(name, qty, unit, k):
    rule = classify(name)
    if rule.type == "discrete":            # œufs : on compare le NOMBRE d'œufs
        return float(scale_eggs(qty, k).whole_eggs)
    if rule.type == "geometric":           # temps : minutes scalées
        return scale_time(qty, k).minutes
    return scale(name, qty, unit, k).value  # linear / sublinear / fixed


def table_corpus(k, types=None, limit=None):
    """Lignes (recette | ingrédient | type | base | ×N naïf | moteur | écart)."""
    rows = []
    for r in ALL:
        t = classify(r["name"]).type
        if types is not None and t not in types:
            continue
        naif = r["qty"] * k
        mot = moteur_corpus(r["name"], r["qty"], r["unit"], k)
        rows.append({
            "recette": r["recipe"].replace(".cook", ""),
            "ingrédient": r["name"],
            "type": t,
            "base": round(r["qty"], 3),
            "×N_naïf": round(naif, 3),
            "moteur": round(mot, 3),
            "écart": round(mot - naif, 3),
        })
    return rows[:limit] if limit else rows


def afficher_corpus(rows):
    """Rendu texte aligné, sans pandas (offline, déterministe)."""
    cols = ["recette", "ingrédient", "type", "base", "×N_naïf", "moteur", "écart"]
    widths = {c: max(len(c), *(len(str(r[c])) for r in rows)) for c in cols}
    header = " | ".join(c.ljust(widths[c]) for c in cols)
    print(header)
    print("-" * len(header))
    for r in rows:
        print(" | ".join(str(r[c]).ljust(widths[c]) for c in cols))

In [ ]:
# ×2 — focus sur les ASSAISONNEMENTS sous-linéaires du corpus : c'est là que le
# moteur diverge du ×N naïf (le naïf sur-sale au doublement). Écart attendu < 0.
print("=== ×2 — sous-linéaires (sel, poivre, ail, herbes...) ===")
rows = table_corpus(2.0, types={"sublinear"})
afficher_corpus(rows)
assert all(r["écart"] < 0 for r in rows), "à ×2, le sous-linéaire doit sous-doser vs le naïf"
print("\nOK — à ×2, tous les sous-linéaires du corpus dosent MOINS que le ×N naïf.")

In [ ]:
# ×2 — le LINÉAIRE égale le ×N naïf (écart nul), tandis que le TEMPS et les ŒUFS
# divergent (lois géométrique / discrète). Échantillon lisible.
print("=== ×2 — linéaires (échantillon : écart nul attendu) ===")
afficher_corpus(table_corpus(2.0, types={"linear"}, limit=6))

print("\n=== ×2 — temps & œufs (divergence non linéaire / discrète) ===")
afficher_corpus(table_corpus(2.0, types={"geometric", "discrete"}))

# Contrôles chiffrés (mêmes asserts que tests/test_scaling_validation.py) :
assert all(abs(r["écart"]) < 1e-9 for r in table_corpus(2.0, types={"linear"})), \
    "le linéaire doit égaler le ×N naïf"
print("\nOK — linéaire == ×N naïf ; temps (×2 -> ~32 min, pas 40) et œufs (3 -> 5, pas 6) divergent.")

In [ ]:
# ÷2 et ×4 — la divergence est cohérente dans les DEUX sens. À ÷2 le moteur
# sur-dose les assaisonnements vs le naïf (qui sous-sale) ; à ×4 il sous-dose
# franchement (le naïf sur-sale x4).
for k in (0.5, 4.0):
    label = "÷2" if k == 0.5 else "×4"
    print(f"\n=== {label} (k={k}) — sous-linéaires ===")
    rows = table_corpus(k, types={"sublinear"})
    afficher_corpus(rows)
    if k > 1:
        assert all(r["écart"] < 0 for r in rows), "k>1 : moteur < naïf"
    else:
        assert all(r["écart"] > 0 for r in rows), "k<1 : moteur > naïf"

# Temps aux trois facteurs : k^(2/3) (≈ table) vs ×N naïf.
print("\n=== Temps de cuisson : moteur (k^2/3) vs ×N naïf ===")
print(f"{'k':>4} | {'naïf (20·k)':>12} | {'moteur':>8} | {'écart':>7}")
for k in (0.5, 2.0, 4.0):
    m = scale_time(20, k).minutes
    print(f"{k:>4} | {20 * k:>12.2f} | {m:>8.2f} | {m - 20 * k:>7.2f}")

## Conclusion — ce que ce jeu alimente (et ne résout pas)

**Lecture des tableaux ci-dessus (corpus S0.6, 13 recettes) :**

| Type | Loi moteur | Écart vs ×N naïf | Raisonnement culinaire |
|------|-----------|------------------|------------------------|
| `linear` (pâtes, tomates, liquides, fromages) | `qty·k` | **nul** | la règle de trois suffit |
| `sublinear` (sel, poivre, ail, herbes) | `qty·k^coeff` (0<coeff<1) | **< 0 si k>1, > 0 si k<1** | le naïf sur-/sous-assaisonne |
| `discrete` (œufs) | unités entières + reste, RZ1 | **< 0 au doublement franc** | 3 œufs ×2 → 5 (liant), pas 6 |
| `geometric` (temps) | `t·k^(2/3)` | **< 0 si k>1** | diffusion surface/volume |
| `fixed` (températures, verrou `=`) | inchangé | **0** | une température ne se multiplie pas |

**Ce jeu ALIMENTE la calibration F2 / RZ2** : il **chiffre**, sur de vraies
recettes, *où* et *de combien* le moteur s'écarte de la multiplication naïve. Ces
écarts sont l'entrée de la boucle **F2** (ajuster les coefficients de
`docs/scaling/table-scaling-sale.json`) et la réponse au risque **R3**
(« coefficients faux »).

**Il ne RÉSOUT PAS RZ2** : les coefficients (0.75 pour le sel, 0.60 pour le
piment, 0.6667 pour le temps...) restent des **valeurs de DÉPART à CALIBRER F2**
(cf. `meta.disclaimer` de la table). **RZ2 demeure `open` (post-v1)** : seule une
campagne de tests réels (×2 / ÷2 en cuisine) pourra les figer. Tant que F2 n'a
pas tranché, ces coefficients ne doivent pas être présentés comme exacts.

📎 Références : `tests/test_scaling_validation.py` (asserts chiffrés équivalents) ;
`docs/recherche-ouverte.md` §C (calibration) ; `docs/architecture.md` §11 (ligne
« Calibration scaling — moteur vs ×N naïf »), §13 R3 ; `table-scaling-sale.json`
→ `meta.disclaimer` ; `docs/stories/E3.md`.